## 1. Introduction

This report walks through a Python implementation of the SIFT (Scale-Invariant Feature Transform) algorithm: from image preprocessing to final 128-D descriptors.  We will see:

1. Base-image generation  
2. Constructing Scale Space
3. Gaussian and DoG pyramids  
4. Keypoint detection, localization, and orientation assignment  
5. Descriptor formation  
6. Practical notes, parameter choices, and complexity considerations  

---

## 2. Global Settings

```python
float_tolerance = 1e-7
```

- A constant used in normalization steps to avoid division by zero.

---

## 3. Base‐Image Generation

In [ ]:
def generateBaseImage(image, sigma, assumed_blur):
    # 1) Upsample by 2× using INTER_LINEAR interpolation
    image = resize(image, (0,0), fx=2, fy=2, interpolation=INTER_LINEAR)

    # 2) Compute extra blur needed:
    #    σ_total² = (2·σ_assumed)² + σ_diff²  →  σ_diff = √(σ² – (2·σ_assumed)²)
    sigma_diff = sqrt(max(sigma**2 - (2*assumed_blur)**2, 0.01))

    # 3) Apply Gaussian blur of σ_diff so that overall blur = σ
    return GaussianBlur(image, (0,0), sigmaX=sigma_diff, sigmaY=sigma_diff)

- **Why upsample?**  
  - Improves localization of small features.  

<div align="center">
  <img src="Data/report_diagrams/upsampling.jpg" alt="My Image" width="400">
</div>

- **Blur principle:**  
  * Blurring with $\sigma_1$ then $\sigma_2$ is equivalent to blurring directly with $\sigma_{\rm total}$, where:
  $$
    \sigma_{\rm total}^2 = \sigma_1^2 + \sigma_2^2
  $$  
  * Assuming the input image is blurred with `sigma_assumed = 0.5`, desired $\sigma_{\rm total}$ = 1.6, we achieve that by applying $\sigma_{\rm diff}$


---

## 4. Octave Count

In [ ]:
def computeNumberOfOctaves(image_shape):
    return int(round(log2(min(image_shape)) - 1))

- Halve the image repeatedly until its smallest side ≲ 4 px.  

---

## 5. Gaussian‐Scale Interleaving

In [ ]:
def generateGaussianKernels(sigma, num_intervals):
    k = 2 ** (1.0 / num_intervals)
    num_images = num_intervals + 3
    kernels = zeros(num_images)
    kernels[0] = sigma
    for idx in range(1, num_images):
        prev = (k ** (idx - 1)) * sigma
        total = k * prev
        kernels[idx] = sqrt(total**2 - prev**2)
    return kernels

- Blurs are spaced by a constant multiplicative factor $k$.  
- For the 1st octave $\sigma$ ranges from $\sigma_{o}$ to $k \times \sigma_{o}$
- **Equations:**  
  - $k = 2^{1/s}$, where $s$ is the number of blurring levels per octave
  - $\sigma_i = \sqrt{(k\sigma_{i-1})^2 - \sigma_{i-1}^2}$

---

## 6. Building the Gaussian Pyramid

In [ ]:
def generateGaussianImages(image, num_octaves, kernels):
    pyramid = []
    for o in range(num_octaves):
        octave_imgs = [image]
        for σ_diff in kernels[1:]:
            image = GaussianBlur(image, (0,0), sigmaX=σ_diff, sigmaY=σ_diff)
            octave_imgs.append(image)
        pyramid.append(octave_imgs)

        # Downsample the image 2 levels before the end
        base = octave_imgs[-3]
        image = resize(base, (base.shape[1]//2, base.shape[0]//2), interpolation=INTER_NEAREST)
    return array(pyramid, dtype=object)

- Each octave holds $s+3$ blurred images (for $s$ intervals).  

<div align="center">
  <img src="Data/report_diagrams/octaves.png" alt="My Image">
</div>

---

## 7. Difference‐of‐Gaussian (DoG) Pyramid

In [ ]:
def generateDoGImages(gaussian_images):
    return array([
        [second - first
         for first, second in zip(octave[:-1], octave[1:])]
        for octave in gaussian_images
    ], dtype=object)

- Approximates a Laplacian‐of‐Gaussian by simple subtraction. 

<div align="center">
  <img src="Data/report_diagrams/SIFT_steps.png" alt="My Image" width="600">
</div>

---

## 8. Keypoint Detection

### 8.1. Identifying Extremal Pixels
- Slide a 3×3 window across each triplet of DoG images.  
- Reject low‐contrast points via a threshold scaled by the number of intervals.

In [ ]:
def isPixelAnExtremum(L, C, U, threshold):
    val = C[1,1]
    if abs(val) <= threshold:
        return False
    if val > 0:
        return (val >= L).all() and (val >= U).all() \
           and (val >= C[0,:]).all() and (val >= C[2,:]).all() \
           and val >= C[1,0] and val >= C[1,2]
    else:
        return (val <= L).all() and (val <= U).all() \
           and (val <= C[0,:]).all() and (val <= C[2,:]).all() \
           and val <= C[1,0] and val <= C[1,2]

### 8.2. Quadratic Localization

```python
def localizeExtremumViaQuadraticFit(...):
```
1) Build 3×3×3 pixel cube around candidate
2) Compute gradient ∇ and Hessian H via central differences:
        ∂f/∂x ≈ (f(x+1) - f(x-1)) / 2
        ∂²f/∂x² ≈ f(x+1) - 2f(x) + f(x-1)
3) Solve Δ = -H⁻¹ ∇ to refine location
4) Reject if:
    * Moves outside image bounds  
    * Low contrast after interpolation
    * Edge‐like: $\frac{\rm trace(H)^2}{\det(H)}$ too large
5) Package into OpenCV KeyPoint with:
    * pt = (x⋅2^octave, y⋅2^octave)  
    * size, response, octave encoding
---

## 9. Orientation Assignment

```python
def computeKeypointsWithOrientations(...):
```
* In a circular window (radius ∝ scale), compute ∇ magnitude & angle at each pixel.
* Weight by a Gaussian centered on the keypoint.
* Build a 36‐bin histogram, smooth it, find peaks ≥ 0.8·max.
* For each peak, interpolate its position for sub‐bin accuracy.
* Ensures rotation invariance by assigning one or more dominant orientations to each keypoint.

<div align="center">
  <img src="Data/report_diagrams/orientations.png" width="400" alt="My Image">
</div>

---

## 10. Descriptor Generation

```python
def generateDescriptors(keypoints, gaussian_images, window_width=4, num_bins=8, ...):
```
* For each keypoint, define a square region aligned with its orientation.
* Divide into 4×4 cells, each with an 8-bin histogram of ∇ directions.
* Trilinearly interpolate contributions across spatial & orientation bins.
* Concatenate to a 128-dim vector, threshold at 0.2·‖v‖, renormalize.

- **Parameter choices:**  
  - σ = 1.6 (base blur)  
  - 3 intervals per octave (therefore 6 DoG images)  
  - Descriptor grid = 4×4 cells × 8 bins = 128-D  

<div align="center">
  <img src="Data/report_diagrams/feature_hist.jpg" alt="My Image">
</div>

---

## 11. Putting It All Together

In [ ]:
def computeKeypointsAndDescriptors(...):
    base  = generateBaseImage(...)
    oct   = computeNumberOfOctaves(base.shape)
    kern  = generateGaussianKernels(sigma, num_intervals)
    g_pyr = generateGaussianImages(base, oct, kern)
    d_pyr = generateDoGImages(g_pyr)
    kps   = findScaleSpaceExtrema(g_pyr, d_pyr, ...)
    kps   = removeDuplicateKeypoints(kps)
    kps   = convertKeypointsToInputImageSize(kps)
    desc  = generateDescriptors(kps, g_pyr)
    return kps, desc


---

## Notes

- **Complexity:**  
  - Building pyramids and scanning each pixel scales roughly as $O(N)$ per octave, with pruning via contrast/edge thresholds.  
  - Descriptor computation is heavier but still linear in the number of detected keypoints.

- **Parameter sensitivity:**  
  - More intervals → finer scale resolution but higher cost.  
  - Contrast threshold trades off repeatability vs. number of keypoints.

- **References:**  
  - D. Lowe, “Distinctive Image Features from Scale‐Invariant Keypoints,” IJCV, 2004.